<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/0904_News_Bot_Projct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 뉴스 요약봇 만들기 [프로젝트]

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 1. 뉴스기사 요약해보기

> 💡 **GPU 런타임 사용을 권장합니다.**

새로운 데이터셋에 대해서 추상적 요약과 추출적 요약을 모두 해보는 시간을 가져봐요.

먼저 주요 라이브러리 버전을 확인해 보죠.

### 답안 구현 기준과 출처

- `Based-On: NLP-01-02-C012@40a30afc8988` — 공백 tokenization(토큰화), word↔ID, `torch.long`, padding(패딩) 흐름
- `Supplemental-Based-On: NLP-01-01-C008@2fe05534e124` — 영어 text cleaning(텍스트 정제)
- `Supplemental-Based-On: NLP-01-02-C011@a69ad5a21a6c` — 길이 분석·filtering(필터링)
- `Supplemental-Based-On: MLDL-03-02-C057@46154b169e1c` — PyTorch training loop(학습 루프)
- `Supplemental-Based-On: MLDL-03-02-C061@1eeb3dd65f47` — model/checkpoint 저장·불러오기
- 직접 레슨 출처: `뉴스 요약봇 만들기.ipynb`, `lms_export_with_user_state`, SHA-256 `2713b140d1edc195bb00ae8b18822dbced6941bdbe516fd9fc1624cb6907a8da`

아래 답안은 레슨의 전처리 → 분할 → 단어장 → padding → Encoder/Decoder/Attention → 학습 → 추론 → Summa 순서를 유지한 `project_extension`입니다. 직접 레슨의 저장 출력과 현재 source가 일부 일치하지 않아 코드는 재사용하되 출력은 실행 근거로 인용하지 않습니다.


> **실행 안내**: Colab의 GPU runtime(런타임)에서 위에서 아래로 실행합니다. 전체 데이터 학습은 오래 걸릴 수 있습니다. 제출 전 `런타임 → 모두 실행` 후 loss graph(손실 그래프)와 비교표 출력이 저장되었는지 확인합니다.

`NEWS_BOT_SMOKE=1`은 구조 검증용 축소 경로입니다. 제출 학습에는 기본 PROJECT 모드를 사용합니다.


In [ ]:
from pathlib import Path
import os
import tempfile

# Local safe smoke(로컬 안전 간이 실행)에서는 Drive와 network(네트워크)를 사용하지 않습니다.
SMOKE_MODE = os.environ.get("NEWS_BOT_SMOKE", "0") == "1"
IN_COLAB = False

if SMOKE_MODE:
    DRIVE_DATA_DIR = Path(
        os.environ.get(
            "NEWS_BOT_SMOKE_DIR",
            str(Path(tempfile.gettempdir()) / "news_bot_smoke"),
        )
    )
else:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        DRIVE_DATA_DIR = Path.cwd() / "news_summarization_data"

if IN_COLAB:
    mount_candidates = [Path("/content/drive"), Path("/content/google_drive")]
    mount_candidates.extend(
        Path(f"/content/google_drive_{number}") for number in range(2, 10)
    )

    # 이미 연결된 Drive를 먼저 찾습니다.
    MOUNT_POINT = next(
        (
            path
            for path in mount_candidates
            if os.path.ismount(path) and (path / "MyDrive").is_dir()
        ),
        None,
    )

    if MOUNT_POINT is None:
        # 기존 파일이 든 mount point는 건드리지 않고, 비어 있거나 없는 경로를 선택합니다.
        def is_available_mountpoint(path):
            if path.is_symlink() or (path.exists() and not path.is_dir()):
                return False
            if not path.exists():
                return True
            try:
                return next(path.iterdir(), None) is None
            except FileNotFoundError:
                return True

        last_mountpoint_error = None
        for candidate in mount_candidates:
            if not is_available_mountpoint(candidate):
                continue
            try:
                drive.mount(str(candidate))
                MOUNT_POINT = candidate
                break
            except ValueError as error:
                if "Mountpoint must" not in str(error):
                    raise
                # 검사 직후 다른 작업이 경로를 채운 race(경쟁)라면 다음 빈 경로로 재시도합니다.
                last_mountpoint_error = error
        if MOUNT_POINT is None:
            raise RuntimeError(
                "사용 가능한 빈 Google Drive mount point가 없습니다."
            ) from last_mountpoint_error

    DRIVE_DATA_DIR = (
        MOUNT_POINT / "MyDrive/Aiffel_EPA/news_summarization/data"
    )

DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"실행 모드: {'SMOKE' if SMOKE_MODE else 'PROJECT'}")
print(f"데이터·체크포인트 경로: {DRIVE_DATA_DIR}")


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "summa": "summa",
    "nltk": "nltk",
    "tqdm": "tqdm",
}
missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    if SMOKE_MODE:
        raise ImportError(
            "safe smoke에서는 package를 설치하지 않습니다. "
            f"먼저 설치해 주세요: {missing_packages}"
        )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages]
    )

print("필수 package 확인 완료")


In [ ]:
from collections import Counter
from copy import deepcopy
from importlib.metadata import version
from uuid import uuid4
import hashlib
import html
import random
import re

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import summa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import display
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if SMOKE_MODE:
    device = torch.device("cpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("nltk:", nltk.__version__)
print("torch:", torch.__version__)
print("pandas:", pd.__version__)
print("summa:", version("summa"))
print("device:", device)


## Step 1. 데이터 수집하기

데이터는 아래 링크에 있는 뉴스 기사 데이터(`news_summary_more.csv`)를 사용하세요.
- [sunnysai12345/News_Summary](https://github.com/sunnysai12345/News_Summary)<br>

아래의 코드로 데이터를 다운로드할 수 있어요.

In [ ]:
from urllib.request import urlretrieve

NEWS_DATA_COMMIT = "a86081060bbb17c5686b478b2f7f18df6e87303d"
EXPECTED_ROW_COUNT = 98_401
news_url = (
    "https://raw.githubusercontent.com/sunnysai12345/News_Summary/"
    f"{NEWS_DATA_COMMIT}/news_summary_more.csv"
)
news_csv_path = DRIVE_DATA_DIR / "news_summary_more.csv"
required_columns = {"text", "headlines"}

def read_and_validate_news_csv(path):
    frame = pd.read_csv(path, encoding="iso-8859-1")
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"필수 column(열)이 없습니다: {sorted(missing_columns)}"
        )
    if len(frame) != EXPECTED_ROW_COUNT:
        raise ValueError(
            f"예상 row 수 {EXPECTED_ROW_COUNT:,}와 다릅니다: {len(frame):,}"
        )
    return frame

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

if SMOKE_MODE:
    cities = ["seoul", "london", "paris", "delhi", "tokyo", "sydney", "berlin", "madrid"]
    topics = ["transit", "health", "energy", "school", "housing", "water", "safety", "culture"]
    rows = []
    for city in cities:
        for topic in topics:
            rows.append(
                {
                    "text": (
                        f"The {city} council approved a new {topic} project on Monday. "
                        f"Officials said the {topic} plan will help local residents. "
                        "Work starts next month after an independent safety review."
                    ),
                    "headlines": f"{city} approves new {topic} project",
                }
            )
    data = pd.DataFrame(rows)
    print(f"synthetic smoke data: {len(data):,}개 sample(샘플)")
else:
    if news_csv_path.is_file() and news_csv_path.stat().st_size > 0:
        print(f"기존 CSV를 사용합니다: {news_csv_path}")
        data = read_and_validate_news_csv(news_csv_path)
    else:
        temp_path = DRIVE_DATA_DIR / f".{news_csv_path.name}.{uuid4().hex}.part"
        try:
            urlretrieve(news_url, temp_path)
            if not temp_path.is_file() or temp_path.stat().st_size == 0:
                raise IOError("CSV 다운로드 결과가 비어 있습니다.")
            data = read_and_validate_news_csv(temp_path)
            os.replace(temp_path, news_csv_path)
        finally:
            temp_path.unlink(missing_ok=True)
        print(f"다운로드·검증·저장 완료: {news_csv_path}")

print(f"불러오기 완료: {len(data):,}개 sample(샘플)")
if not SMOKE_MODE:
    print("dataset commit:", NEWS_DATA_COMMIT)
    print("dataset SHA-256:", file_sha256(news_csv_path))


In [ ]:
data.sample(min(10, len(data)), random_state=SEED)


이 데이터는 기사의 본문에 해당되는 text와 headlines 두 가지 열로 구성되어 있습니다.<br>
추상적 요약을 하는 경우에는 text를 본문, headlines를 이미 요약된 데이터로 삼아서 모델을 학습할 수 있어요. 추출적 요약을 하는 경우에는 오직 text열만을 사용하세요.

## Step 2. 데이터 전처리하기 (추상적 요약)

실습에서 사용된 전처리를 참고하여 각자 필요하다고 생각하는 전처리를 추가 사용하여 텍스트를 정규화 또는 정제해 보세요. 만약, 불용어 제거를 선택한다면 상대적으로 길이가 짧은 요약 데이터에 대해서도 불용어를 제거하는 것이 좋을지 고민해 보세요.

### 답안 2-1. 분석, 중복·결측 제거

프로젝트의 `text/headlines`를 레슨의 `Text/Summary` 이름으로 한 번 연결합니다. Summa와 최종 비교에는 정제 전 문장이 필요하므로 `OriginalText`, `OriginalSummary`를 함께 보존합니다.


In [ ]:
print("원본 shape:", data.shape)
print("column별 NULL:\n", data[["text", "headlines"]].isnull().sum())
print("text unique:", data["text"].nunique())
print("headlines unique:", data["headlines"].nunique())

prepared = (
    data[["text", "headlines"]]
    .rename(columns={"text": "Text", "headlines": "Summary"})
    .drop_duplicates(subset=["Text"])
    .dropna(subset=["Text", "Summary"])
    .reset_index(names="source_row")
)
prepared["OriginalText"] = prepared["Text"].astype(str)
prepared["OriginalSummary"] = prepared["Summary"].astype(str)

print("중복·NULL 제거 후 shape:", prepared.shape)
prepared.head(3)


### 답안 2-2. normalization(정규화)과 cleaning(정제)

레슨처럼 소문자화, HTML 제거, 괄호·특수문자 제거, contraction(축약형) 확장을 적용합니다. 본문은 핵심어 밀도를 높이기 위해 stopwords(불용어)를 제거하고, 짧은 정답 요약은 문법 정보를 보존하기 위해 불용어를 남깁니다.


In [ ]:
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    if SMOKE_MODE:
        print("SMOKE_MODE: 내장된 최소 stopword set을 사용합니다.")
    else:
        if not nltk.download("stopwords", quiet=True):
            raise RuntimeError("NLTK stopwords 다운로드에 실패했습니다.")

try:
    from nltk.corpus import stopwords
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    STOP_WORDS = {
        "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
        "has", "he", "in", "is", "it", "its", "of", "on", "that", "the",
        "to", "was", "were", "will", "with",
    }

print("stopword 수:", len(STOP_WORDS))


In [ ]:
CONTRACTIONS = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot",
    "could've": "could have", "couldn't": "could not", "didn't": "did not",
    "doesn't": "does not", "don't": "do not", "hadn't": "had not",
    "hasn't": "has not", "haven't": "have not", "he'd": "he would",
    "he'll": "he will", "he's": "he is", "how'd": "how did",
    "how'll": "how will", "how's": "how is", "i'd": "i would",
    "i'll": "i will", "i'm": "i am", "i've": "i have",
    "isn't": "is not", "it'd": "it would", "it'll": "it will",
    "it's": "it is", "let's": "let us", "mayn't": "may not",
    "might've": "might have", "mightn't": "might not", "must've": "must have",
    "mustn't": "must not", "needn't": "need not", "oughtn't": "ought not",
    "shan't": "shall not", "she'd": "she would", "she'll": "she will",
    "she's": "she is", "should've": "should have", "shouldn't": "should not",
    "that's": "that is", "there'd": "there would", "there's": "there is",
    "they'd": "they would", "they'll": "they will", "they're": "they are",
    "they've": "they have", "wasn't": "was not", "we'd": "we would",
    "we'll": "we will", "we're": "we are", "we've": "we have",
    "weren't": "were not", "what'll": "what will", "what're": "what are",
    "what's": "what is", "what've": "what have", "when's": "when is",
    "where'd": "where did", "where's": "where is", "who'll": "who will",
    "who's": "who is", "who've": "who have", "why's": "why is",
    "will've": "will have", "won't": "will not", "would've": "would have",
    "wouldn't": "would not", "y'all": "you all", "you'd": "you would",
    "you'll": "you will", "you're": "you are", "you've": "you have",
}

HTML_TAG_RE = re.compile(r"<[^>]*>")
PAREN_RE = re.compile(r"\([^)]*\)")
POSSESSIVE_RE = re.compile(r"'s\b")
NON_ALPHA_RE = re.compile(r"[^a-zA-Z]")
REPEATED_M_RE = re.compile(r"m{3,}")

def preprocess_sentence(sentence, remove_stopwords=True):
    sentence = html.unescape(str(sentence).lower())
    sentence = HTML_TAG_RE.sub(" ", sentence)
    sentence = PAREN_RE.sub(" ", sentence)
    sentence = sentence.replace('"', " ")
    sentence = " ".join(
        CONTRACTIONS.get(token, token) for token in sentence.split()
    )
    sentence = POSSESSIVE_RE.sub("", sentence)
    sentence = NON_ALPHA_RE.sub(" ", sentence)
    sentence = REPEATED_M_RE.sub("mm", sentence)
    words = sentence.split()

    if remove_stopwords:
        words = [
            word for word in words
            if word not in STOP_WORDS and len(word) > 1
        ]
    else:
        words = [word for word in words if len(word) > 1]
    return " ".join(words)

sample_text = "Everything I bought was great; it wasn't late.<br />My family liked it."
sample_summary = "It wasn't late!"
print("Text:", preprocess_sentence(sample_text))
print("Summary:", preprocess_sentence(sample_summary, remove_stopwords=False))


In [ ]:
clean_text = [
    preprocess_sentence(sentence, remove_stopwords=True)
    for sentence in tqdm(prepared["Text"], desc="Text cleaning")
]
clean_summary = [
    preprocess_sentence(sentence, remove_stopwords=False)
    for sentence in tqdm(prepared["Summary"], desc="Summary cleaning")
]

prepared["Text"] = clean_text
prepared["Summary"] = clean_summary
prepared[["Text", "Summary"]] = prepared[["Text", "Summary"]].replace("", np.nan)
prepared = (
    prepared
    .dropna(subset=["Text", "Summary"])
    .drop_duplicates(subset=["Text"])
    .reset_index(drop=True)
)

print("정제 후 sample 수:", len(prepared))
display(prepared[["Text", "Summary"]].head(3))


### 답안 2-3. 길이 분석과 paired filtering(쌍 필터링)

레슨 값인 본문 50단어, 정답 요약 8단어를 유지합니다. 요약에 SOS/EOS를 붙일 때 한 칸이 더 필요하므로 decoder width(디코더 폭)는 `8 + 1 = 9`로 분리합니다. 하나의 boolean mask(불리언 마스크)로 행을 함께 걸러 source/target 정렬을 보존합니다.


In [ ]:
text_len = prepared["Text"].str.split().str.len()
summary_len = prepared["Summary"].str.split().str.len()

length_stats = pd.DataFrame(
    {
        "Text": text_len.describe(),
        "Summary": summary_len.describe(),
    }
)
display(length_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(text_len, bins=40)
axes[0].set(title="Text length", xlabel="words", ylabel="samples")
axes[1].hist(summary_len, bins=30)
axes[1].set(title="Summary length", xlabel="words", ylabel="samples")
plt.tight_layout()
plt.show()


In [ ]:
TEXT_MAX_LEN = 50
SUMMARY_MAX_WORDS = 8
DECODER_MAX_LEN = SUMMARY_MAX_WORDS + 1

length_mask = (
    prepared["Text"].str.split().str.len().between(1, TEXT_MAX_LEN)
    & prepared["Summary"].str.split().str.len().between(1, SUMMARY_MAX_WORDS)
)
data_preprocessed = prepared.loc[length_mask].reset_index(drop=True)

if len(data_preprocessed) < 2:
    raise ValueError("train/validation 분할에 필요한 sample이 부족합니다.")

rng = np.random.default_rng(SEED)
shuffled_indices = rng.permutation(len(data_preprocessed))
validation_size = max(1, int(len(shuffled_indices) * 0.2))
validation_size = min(validation_size, len(shuffled_indices) - 1)

validation_indices = shuffled_indices[:validation_size]
train_indices = shuffled_indices[validation_size:]
train_df = data_preprocessed.iloc[train_indices].reset_index(drop=True)
validation_df = data_preprocessed.iloc[validation_indices].reset_index(drop=True)

assert set(train_df["source_row"]).isdisjoint(validation_df["source_row"])

def dataframe_sha256(frame, columns):
    digest = hashlib.sha256()
    for row in frame[list(columns)].itertuples(index=False, name=None):
        for value in row:
            encoded = str(value).encode("utf-8")
            digest.update(len(encoded).to_bytes(8, "big"))
            digest.update(encoded)
    return digest.hexdigest()

DATA_FINGERPRINT = dataframe_sha256(
    data_preprocessed, ["source_row", "Text", "Summary"]
)
SPLIT_FINGERPRINT = hashlib.sha256(
    (
        "train:"
        + ",".join(map(str, train_df["source_row"]))
        + "|validation:"
        + ",".join(map(str, validation_df["source_row"]))
    ).encode("utf-8")
).hexdigest()

print("길이 filtering 후:", len(data_preprocessed))
print("train:", len(train_df), "validation:", len(validation_df))
print(
    f"Text ≤ {TEXT_MAX_LEN}: {length_mask.mean():.2%}, "
    f"Summary ≤ {SUMMARY_MAX_WORDS} 조건 포함"
)
print("data fingerprint:", DATA_FINGERPRINT[:16])
print("split fingerprint:", SPLIT_FINGERPRINT[:16])


### 답안 2-4. vocabulary(단어장), integer encoding(정수 인코딩), padding

데이터 누수(leakage)를 막기 위해 단어장은 train split(학습 분할)에서만 만듭니다. `<pad>`와 `<unk>`를 분리하고 target에는 `sostoken`, `eostoken`을 예약합니다. 레슨의 희귀 단어 threshold(임곗값)는 분포 진단에 사용하고, 실제 단어장은 레슨 값인 상위 8,000/2,000개로 제한합니다.


In [ ]:
SRC_VOCAB_LIMIT = 8_000
TAR_VOCAB_LIMIT = 2_000
SRC_RARE_DIAGNOSTIC_THRESHOLD = 7
TAR_RARE_DIAGNOSTIC_THRESHOLD = 6

def word_counter(texts):
    counter = Counter()
    for sentence in texts:
        counter.update(sentence.split())
    return counter

def rare_word_report(counter, threshold, label):
    total_count = len(counter)
    total_frequency = sum(counter.values())
    rare_count = sum(count < threshold for count in counter.values())
    rare_frequency = sum(
        count for count in counter.values() if count < threshold
    )
    print(f"[{label}] 전체 단어 수: {total_count:,}")
    print(f"[{label}] {threshold}회 미만 희귀 단어 수: {rare_count:,}")
    print(
        f"[{label}] 전체 빈도 중 희귀 단어 빈도: "
        f"{rare_frequency / max(total_frequency, 1):.2%}"
    )

def build_limited_vocab(texts, max_size, special_tokens):
    counter = word_counter(texts)
    vocab = {token: index for index, token in enumerate(special_tokens)}
    for word, _ in counter.most_common(max_size - len(vocab)):
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab

src_counter = word_counter(train_df["Text"])
tar_counter = word_counter(train_df["Summary"])
# 레슨의 threshold는 분포 진단에 사용하고, 실제 vocab은 레슨처럼 상위 크기로 제한합니다.
rare_word_report(src_counter, SRC_RARE_DIAGNOSTIC_THRESHOLD, "Text")
rare_word_report(tar_counter, TAR_RARE_DIAGNOSTIC_THRESHOLD, "Summary")

src_vocab = build_limited_vocab(
    train_df["Text"], SRC_VOCAB_LIMIT, ["<pad>", "<unk>"]
)
tar_vocab = build_limited_vocab(
    train_df["Summary"],
    TAR_VOCAB_LIMIT,
    ["<pad>", "<unk>", "sostoken", "eostoken"],
)

SRC_PAD_ID, SRC_UNK_ID = src_vocab["<pad>"], src_vocab["<unk>"]
TAR_PAD_ID, TAR_UNK_ID = tar_vocab["<pad>"], tar_vocab["<unk>"]
SOS_ID, EOS_ID = tar_vocab["sostoken"], tar_vocab["eostoken"]
src_index_to_word = {index: word for word, index in src_vocab.items()}
tar_index_to_word = {index: word for word, index in tar_vocab.items()}

print("source vocab:", len(src_vocab), "target vocab:", len(tar_vocab))


In [ ]:
def encode_source(texts):
    return [
        [src_vocab.get(word, SRC_UNK_ID) for word in text.split()]
        for text in texts
    ]

def encode_target(texts):
    decoder_inputs, decoder_targets = [], []
    for text in texts:
        content_ids = [
            tar_vocab.get(word, TAR_UNK_ID) for word in text.split()
        ][:SUMMARY_MAX_WORDS]
        decoder_inputs.append([SOS_ID, *content_ids])
        decoder_targets.append([*content_ids, EOS_ID])
    return decoder_inputs, decoder_targets

def pad_to_tensor(sequences, max_len, padding_value):
    tensor = torch.full(
        (len(sequences), max_len),
        padding_value,
        dtype=torch.long,
    )
    for row, sequence in enumerate(sequences):
        clipped = sequence[:max_len]
        if clipped:
            tensor[row, :len(clipped)] = torch.tensor(clipped, dtype=torch.long)
    return tensor

train_src_sequences = encode_source(train_df["Text"])
validation_src_sequences = encode_source(validation_df["Text"])
train_decoder_inputs, train_decoder_targets = encode_target(train_df["Summary"])
validation_decoder_inputs, validation_decoder_targets = encode_target(
    validation_df["Summary"]
)

encoder_input_train = pad_to_tensor(
    train_src_sequences, TEXT_MAX_LEN, SRC_PAD_ID
)
encoder_input_validation = pad_to_tensor(
    validation_src_sequences, TEXT_MAX_LEN, SRC_PAD_ID
)
decoder_input_train = pad_to_tensor(
    train_decoder_inputs, DECODER_MAX_LEN, TAR_PAD_ID
)
decoder_target_train = pad_to_tensor(
    train_decoder_targets, DECODER_MAX_LEN, TAR_PAD_ID
)
decoder_input_validation = pad_to_tensor(
    validation_decoder_inputs, DECODER_MAX_LEN, TAR_PAD_ID
)
decoder_target_validation = pad_to_tensor(
    validation_decoder_targets, DECODER_MAX_LEN, TAR_PAD_ID
)

assert encoder_input_train.dtype == torch.long
assert encoder_input_train.shape == (len(train_df), TEXT_MAX_LEN)
assert decoder_input_train.shape == (len(train_df), DECODER_MAX_LEN)
assert torch.all(decoder_input_train[:, 0] == SOS_ID)
assert torch.all((decoder_target_train == EOS_ID).any(dim=1))

print("encoder train:", tuple(encoder_input_train.shape), encoder_input_train.dtype)
print("decoder input:", tuple(decoder_input_train.shape))
print("decoder target:", tuple(decoder_target_train.shape))


## Step 3. 어텐션 메커니즘 사용하기 (추상적 요약)

일반적인 seq2seq보다는 어텐션 메커니즘을 사용한 seq2seq를 사용하는 것이 더 나은 성능을 얻을 수 있어요. 실습 내용을 참고하여 어텐션 메커니즘을 사용한 seq2seq를 설계해 보세요.

### 답안 3-1. Encoder–Decoder와 Luong dot Attention

레슨의 3-layer LSTM, embedding 128, hidden 256, dropout 0.4를 PROJECT 모드에서 유지합니다. `pack_padded_sequence`로 Encoder의 PAD 영향을 제거하고, Attention score(어텐션 점수)에도 source padding mask(원문 패딩 마스크)를 적용합니다.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, pad_id, num_layers=3, dropout=0.4):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=pad_id
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )

    def forward(self, source):
        lengths = source.ne(self.pad_id).sum(dim=1).clamp(min=1).cpu()
        embedded = self.embedding(source)
        packed = pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        packed_outputs, (hidden, cell) = self.lstm(packed)
        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=source.size(1),
        )
        return outputs, hidden, cell


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, pad_id, num_layers=3, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=pad_id
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )

    def forward(self, target, hidden, cell):
        embedded = self.embedding(target)
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        return outputs, hidden, cell


class LuongDotAttention(nn.Module):
    def forward(self, decoder_outputs, encoder_outputs, encoder_mask):
        scores = torch.bmm(
            decoder_outputs, encoder_outputs.transpose(1, 2)
        )
        scores = scores.masked_fill(
            ~encoder_mask.unsqueeze(1),
            torch.finfo(scores.dtype).min,
        )
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, encoder_outputs)
        return context, weights


class Seq2SeqWithAttention(nn.Module):
    def __init__(self, encoder, decoder, target_vocab_size, hidden_size, source_pad_id):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.attention = LuongDotAttention()
        self.concat = nn.Linear(hidden_size * 2, hidden_size)
        self.output_layer = nn.Linear(hidden_size, target_vocab_size)
        self.source_pad_id = source_pad_id

    def forward(self, encoder_input, decoder_input):
        encoder_outputs, hidden, cell = self.encoder(encoder_input)
        decoder_outputs, _, _ = self.decoder(decoder_input, hidden, cell)
        encoder_mask = encoder_input.ne(self.source_pad_id)
        context, _ = self.attention(
            decoder_outputs, encoder_outputs, encoder_mask
        )
        combined = torch.tanh(
            self.concat(torch.cat((decoder_outputs, context), dim=-1))
        )
        return self.output_layer(combined)


In [ ]:
EMBEDDING_DIM = 16 if SMOKE_MODE else 128
HIDDEN_SIZE = 32 if SMOKE_MODE else 256
NUM_LAYERS = 1 if SMOKE_MODE else 3
DROPOUT = 0.0 if SMOKE_MODE else 0.4
BATCH_SIZE = 8 if SMOKE_MODE else 256
EPOCHS = 2 if SMOKE_MODE else 50
LEARNING_RATE = 0.001
PATIENCE = 2
MODEL_CONFIG = {
    "embedding_dim": EMBEDDING_DIM,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "text_max_len": TEXT_MAX_LEN,
    "decoder_max_len": DECODER_MAX_LEN,
}

encoder = Encoder(
    len(src_vocab), EMBEDDING_DIM, HIDDEN_SIZE, SRC_PAD_ID,
    num_layers=NUM_LAYERS, dropout=DROPOUT,
)
decoder = Decoder(
    len(tar_vocab), EMBEDDING_DIM, HIDDEN_SIZE, TAR_PAD_ID,
    num_layers=NUM_LAYERS, dropout=DROPOUT,
)
model = Seq2SeqWithAttention(
    encoder, decoder, len(tar_vocab), HIDDEN_SIZE, SRC_PAD_ID
).to(device)

train_dataset = TensorDataset(
    encoder_input_train, decoder_input_train, decoder_target_train
)
validation_dataset = TensorDataset(
    encoder_input_validation,
    decoder_input_validation,
    decoder_target_validation,
)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

criterion = nn.CrossEntropyLoss(ignore_index=TAR_PAD_ID)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(model)


In [ ]:
smoke_size = min(2, len(encoder_input_train))
model.eval()
with torch.no_grad():
    smoke_logits = model(
        encoder_input_train[:smoke_size].to(device),
        decoder_input_train[:smoke_size].to(device),
    )

expected_shape = (smoke_size, DECODER_MAX_LEN, len(tar_vocab))
assert tuple(smoke_logits.shape) == expected_shape
assert tuple(decoder_target_train[:smoke_size].shape) == (
    smoke_size, DECODER_MAX_LEN
)
print("forward shape:", tuple(smoke_logits.shape))
model.train()
del smoke_logits


### 답안 3-2. 학습, validation(검증), Early Stopping

validation loss가 좋아질 때마다 완성된 checkpoint를 임시 파일에 쓴 뒤 교체합니다. 학습이 중단되어도 기존 checkpoint는 유지되며, 종료 시 가장 낮은 validation loss의 weights를 복원합니다. 재개 시 vocabulary·model config·seed·data/split fingerprint를 검사합니다. weights와 optimizer는 이어지지만 RNG state(난수 상태)는 새로 seed되므로 bit-identical(비트 단위 동일) 실행을 의미하지 않습니다.


In [ ]:
CHECKPOINT_PATH = DRIVE_DATA_DIR / "news_seq2seq_attention_checkpoint.pt"

def train_one_epoch(model, loader, loss_fn, optimizer, device, max_grad_norm=1.0):
    if len(loader) == 0:
        raise ValueError("train loader가 비어 있습니다.")
    model.train()
    total_loss = 0.0

    for encoder_batch, decoder_batch, target_batch in loader:
        encoder_batch = encoder_batch.to(device, non_blocking=True)
        decoder_batch = decoder_batch.to(device, non_blocking=True)
        target_batch = target_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(encoder_batch, decoder_batch)
        loss = loss_fn(
            logits.reshape(-1, logits.size(-1)),
            target_batch.reshape(-1),
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_loss(model, loader, loss_fn, device):
    if len(loader) == 0:
        raise ValueError("validation loader가 비어 있습니다.")
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for encoder_batch, decoder_batch, target_batch in loader:
            encoder_batch = encoder_batch.to(device, non_blocking=True)
            decoder_batch = decoder_batch.to(device, non_blocking=True)
            target_batch = target_batch.to(device, non_blocking=True)
            logits = model(encoder_batch, decoder_batch)
            total_loss += loss_fn(
                logits.reshape(-1, logits.size(-1)),
                target_batch.reshape(-1),
            ).item()
    return total_loss / len(loader)


def save_checkpoint_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f".{path.name}.{uuid4().hex}.part")
    try:
        torch.save(payload, temp_path)
        if not temp_path.is_file() or temp_path.stat().st_size == 0:
            raise IOError("checkpoint 임시 파일이 비어 있습니다.")
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)


def checkpoint_payload(epoch, model, optimizer, history, best_val_loss):
    return {
        "epoch": int(epoch),
        "model_state": {
            key: value.detach().cpu() for key, value in model.state_dict().items()
        },
        "optimizer_state": optimizer.state_dict(),
        "history": {
            key: [float(value) for value in values]
            for key, values in history.items()
        },
        "best_val_loss": float(best_val_loss),
        "src_vocab": src_vocab,
        "tar_vocab": tar_vocab,
        "model_config": MODEL_CONFIG,
        "seed": SEED,
        "data_fingerprint": DATA_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
    }


def train_model(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    total_epochs,
    patience,
    start_epoch=0,
    initial_history=None,
    initial_best_val_loss=float("inf"),
):
    history = initial_history or {"train_loss": [], "validation_loss": []}
    best_val_loss = float(initial_best_val_loss)
    best_state = deepcopy(model.state_dict())
    early_stop_counter = 0
    checkpoint_saved_this_run = False
    checkpoint_error = None

    try:
        for epoch in range(start_epoch, total_epochs):
            train_loss = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )
            validation_loss = evaluate_loss(
                model, validation_loader, criterion, device
            )
            history["train_loss"].append(train_loss)
            history["validation_loss"].append(validation_loss)
            print(
                f"Epoch {epoch + 1:02d}/{total_epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Validation Loss: {validation_loss:.4f}"
            )

            if validation_loss < best_val_loss:
                best_val_loss = validation_loss
                best_state = deepcopy(model.state_dict())
                early_stop_counter = 0
                payload = checkpoint_payload(
                    epoch + 1, model, optimizer, history, best_val_loss
                )
                try:
                    save_checkpoint_atomic(CHECKPOINT_PATH, payload)
                    checkpoint_saved_this_run = True
                    checkpoint_error = None
                    print(f"  best checkpoint 저장: {CHECKPOINT_PATH}")
                except (OSError, RuntimeError) as error:
                    checkpoint_error = str(error)
                    print(f"  checkpoint 저장 경고: {error}")
            else:
                early_stop_counter += 1

            if early_stop_counter >= patience:
                print(f"Early stopping: epoch {epoch + 1}")
                break
    except KeyboardInterrupt:
        model.load_state_dict(best_state)
        print("학습 취소: 메모리의 best weights를 복원했습니다.")
        raise

    model.load_state_dict(best_state)
    return (
        history,
        best_val_loss,
        checkpoint_saved_this_run,
        checkpoint_error,
    )


In [ ]:
RESUME_FROM_CHECKPOINT = False
start_epoch = 0
initial_history = {"train_loss": [], "validation_loss": []}
initial_best_val_loss = float("inf")

if RESUME_FROM_CHECKPOINT:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"checkpoint가 없습니다: {CHECKPOINT_PATH}")
    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location=device, weights_only=True
    )
    if checkpoint["src_vocab"] != src_vocab or checkpoint["tar_vocab"] != tar_vocab:
        raise ValueError("현재 vocabulary와 checkpoint vocabulary가 다릅니다.")
    expected_context = {
        "model_config": MODEL_CONFIG,
        "seed": SEED,
        "data_fingerprint": DATA_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
    }
    context_mismatches = [
        key
        for key, expected_value in expected_context.items()
        if checkpoint.get(key) != expected_value
    ]
    if context_mismatches:
        raise ValueError(
            "checkpoint 실행 조건이 현재 notebook과 다릅니다: "
            f"{context_mismatches}"
        )
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = int(checkpoint["epoch"])
    initial_history = checkpoint["history"]
    initial_best_val_loss = float(checkpoint["best_val_loss"])
    print(f"epoch {start_epoch}의 weights·optimizer부터 이어서 학습합니다.")
    print("RNG state는 새로 seed되어 bit-identical 재현은 보장하지 않습니다.")


In [ ]:
(
    history,
    best_validation_loss,
    checkpoint_saved_this_run,
    checkpoint_error,
) = train_model(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    total_epochs=EPOCHS,
    patience=PATIENCE,
    start_epoch=start_epoch,
    initial_history=initial_history,
    initial_best_val_loss=initial_best_val_loss,
)
print(f"best validation loss: {best_validation_loss:.4f}")
print("checkpoint saved this run:", checkpoint_saved_this_run)
if checkpoint_error:
    print("checkpoint error:", checkpoint_error)


In [ ]:
epochs_axis = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs_axis, history["train_loss"], marker="o", label="Train Loss")
plt.plot(
    epochs_axis,
    history["validation_loss"],
    marker="o",
    label="Validation Loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Step 4. 실제 결과와 요약문 비교하기 (추상적 요약)

원래의 요약문(headlines 열)과 학습을 통해 얻은 추상적 요약의 결과를 비교해 보세요.

### 답안 4-1. greedy decoding(탐욕적 디코딩)과 정답 비교

Encoder를 한 번 실행한 뒤 `sostoken`부터 한 단어씩 생성합니다. `eostoken`에서 멈추며 PAD/SOS는 출력에서 제외합니다.


In [ ]:
def decode_sequence(input_sequence, model, max_len=DECODER_MAX_LEN):
    model.eval()
    if input_sequence.ndim == 1:
        input_sequence = input_sequence.unsqueeze(0)
    input_sequence = input_sequence.to(device)
    decoded_tokens = []

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(input_sequence)
        encoder_mask = input_sequence.ne(model.source_pad_id)
        target = torch.tensor([[SOS_ID]], dtype=torch.long, device=device)

        for _ in range(max_len - 1):
            decoder_output, hidden, cell = model.decoder(
                target, hidden, cell
            )
            context, _ = model.attention(
                decoder_output, encoder_outputs, encoder_mask
            )
            combined = torch.tanh(
                model.concat(torch.cat((decoder_output, context), dim=-1))
            )
            logits = model.output_layer(combined)
            predicted_id = int(logits[0, -1].argmax().item())

            if predicted_id in {EOS_ID, TAR_PAD_ID}:
                break
            if predicted_id != SOS_ID:
                decoded_tokens.append(
                    tar_index_to_word.get(predicted_id, "<unk>")
                )
            target = torch.tensor(
                [[predicted_id]], dtype=torch.long, device=device
            )

    return " ".join(decoded_tokens)


def keyword_overlap_f1(reference, prediction):
    reference_tokens = preprocess_sentence(
        reference, remove_stopwords=True
    ).split()
    prediction_tokens = preprocess_sentence(
        prediction, remove_stopwords=True
    ).split()
    if not reference_tokens or not prediction_tokens:
        return 0.0
    reference_counts = Counter(reference_tokens)
    prediction_counts = Counter(prediction_tokens)
    overlap = sum((reference_counts & prediction_counts).values())
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / max(precision + recall, 1e-12)


In [ ]:
evaluation_count = min(10, len(validation_df))
evaluation_rng = np.random.default_rng(SEED)
evaluation_indices = sorted(
    evaluation_rng.choice(
        len(validation_df), size=evaluation_count, replace=False
    ).tolist()
)

abstractive_rows = []
for index in evaluation_indices:
    prediction = decode_sequence(encoder_input_validation[index], model)
    actual = validation_df.loc[index, "OriginalSummary"]
    actual_keywords = set(
        preprocess_sentence(actual, remove_stopwords=True).split()
    )
    predicted_keywords = set(
        preprocess_sentence(prediction, remove_stopwords=True).split()
    )
    abstractive_rows.append(
        {
            "row": int(validation_df.loc[index, "source_row"]),
            "원문": validation_df.loc[index, "OriginalText"],
            "실제 요약": actual,
            "추상적 요약": prediction,
            "공통 핵심어": ", ".join(sorted(actual_keywords & predicted_keywords)),
            "핵심어 overlap F1": round(
                keyword_overlap_f1(actual, prediction), 3
            ),
        }
    )

abstractive_comparison = pd.DataFrame(abstractive_rows)
pd.set_option("display.max_colwidth", 160)
display(abstractive_comparison)


비교 시 다음 두 축을 봅니다.

1. **문법 완성도**: 주어·동사 관계, 반복, 지나치게 짧거나 끊긴 표현을 확인합니다.
2. **핵심 단어 포함**: `공통 핵심어`와 `핵심어 overlap F1`을 함께 봅니다. 이 값은 stopword 제거 후 unigram(단일 단어) 겹침을 계산한 간단한 진단값이며 표준 ROUGE 점수가 아닙니다. 동의어와 paraphrase(바꿔쓰기)는 표면 일치 점수에 잡히지 않을 수 있습니다.


## Step 5. Summa를 이용해서 추출적 요약해보기

### 답안 5-1. Summa extractive summarization(추출적 요약)

정제 전 `OriginalText`에서 여러 문장으로 된 긴 validation sample을 고릅니다. `ratio` 결과가 비면 Summa의 `words` 방식으로 한 번 더 시도하며, 두 호출이 모두 빈 문자열이면 그대로 표시합니다.


In [ ]:
from summa.summarizer import summarize

def summa_extract(text):
    text = str(text).strip()
    if not text:
        return ""
    for kwargs in ({"ratio": 0.5}, {"words": 30}):
        try:
            result = summarize(text, **kwargs).strip()
        except ValueError:
            result = ""
        if result:
            return result
    return ""


def sentence_count(text):
    return len([part for part in re.split(r"[.!?]+", str(text)) if part.strip()])

candidate_order = sorted(
    range(len(validation_df)),
    key=lambda index: (
        sentence_count(validation_df.loc[index, "OriginalText"]),
        len(str(validation_df.loc[index, "OriginalText"]).split()),
    ),
    reverse=True,
)
summa_indices = candidate_order[:min(10, len(candidate_order))]


In [ ]:
comparison_rows = []
for index in summa_indices:
    article = validation_df.loc[index, "OriginalText"]
    reference = validation_df.loc[index, "OriginalSummary"]
    abstractive = decode_sequence(
        encoder_input_validation[index], model
    )
    extractive = summa_extract(article)

    reference_keywords = set(
        preprocess_sentence(reference, remove_stopwords=True).split()
    )
    abstractive_keywords = set(
        preprocess_sentence(abstractive, remove_stopwords=True).split()
    )
    extractive_keywords = set(
        preprocess_sentence(extractive, remove_stopwords=True).split()
    )

    comparison_rows.append(
        {
            "row": int(validation_df.loc[index, "source_row"]),
            "실제 요약": reference,
            "추상적 요약": abstractive,
            "추출적 요약(Summa)": extractive or "[빈 결과]",
            "추상-핵심어": ", ".join(
                sorted(reference_keywords & abstractive_keywords)
            ),
            "추출-핵심어": ", ".join(
                sorted(reference_keywords & extractive_keywords)
            ),
            "추상-문법 점검": (
                "사람 검토 필요" if len(abstractive.split()) >= 3 else "너무 짧음/빈 결과"
            ),
            "추출-문법 점검": (
                "원문 문장 구조 보존" if extractive else "Summa 빈 결과"
            ),
            "추상 overlap F1": round(
                keyword_overlap_f1(reference, abstractive), 3
            ),
            "추출 overlap F1": round(
                keyword_overlap_f1(reference, extractive), 3
            ),
        }
    )

final_comparison = pd.DataFrame(comparison_rows)
display(final_comparison)

method_summary = pd.DataFrame(
    [
        {
            "방법": "Abstractive (Seq2Seq+Attention)",
            "문법 완성도 관찰": "새 문장을 생성하므로 반복·누락을 표에서 직접 확인",
            "핵심 단어 포함": f"평균 overlap F1 = {final_comparison['추상 overlap F1'].mean():.3f}",
            "특징": "짧고 재구성된 headline 생성 가능",
        },
        {
            "방법": "Extractive (Summa)",
            "문법 완성도 관찰": "원문 문장을 가져오므로 문장 구조가 보존됨",
            "핵심 단어 포함": f"평균 overlap F1 = {final_comparison['추출 overlap F1'].mean():.3f}",
            "특징": "원문보다 짧지 않거나 빈 결과가 생길 수 있음",
        },
    ]
)
display(method_summary)


### 결과 해석

- Abstractive 방식은 원문에 없는 연결을 만들 수 있지만, 학습이 충분하지 않으면 반복·`<unk>`·짧은 출력이 나타납니다.
- Summa는 선택된 원문 문장의 문법을 보존하지만 headline처럼 짧게 바꾸어 쓰지 않으며, 짧은 기사에서는 빈 결과가 생길 수 있습니다.
- 최종 판단은 위 표의 실제 출력에서 문법과 핵심 단어를 함께 확인합니다. 실행하지 않은 상태에서 성능 향상을 미리 단정하지 않습니다.


In [ ]:
completion_checks = pd.Series(
    {
        "paired preprocessing rows": len(data_preprocessed),
        "train tensor shape": tuple(encoder_input_train.shape),
        "validation tensor shape": tuple(encoder_input_validation.shape),
        "best validation loss": float(best_validation_loss),
        "abstractive comparison rows": len(abstractive_comparison),
        "extractive comparison rows": len(final_comparison),
        "checkpoint saved this run": checkpoint_saved_this_run,
        "checkpoint available": CHECKPOINT_PATH.is_file(),
        "checkpoint error": checkpoint_error or "",
    },
    name="project completion check",
)
display(completion_checks)


> **제출 직전 확인**: PROJECT 모드에서 위에서 아래로 실행한 뒤, train/validation loss 그래프가 저장되었는지, 실제·추상·Summa 비교표에 결과가 표시되는지, notebook 출력과 checkpoint가 저장되었는지 확인합니다.


## 프로젝트 제출

## 루브릭

- Abstractive 모델 구성을 위한 텍스트 전처리 단계가 체계적으로 진행되었다.
    - 분석단계, 정제단계, 정규화와 불용어 제거, 데이터셋 분리, 인코딩 과정이 빠짐없이 체계적으로 진행되었다.
- 텍스트 요약모델이 성공적으로 학습되었음을 확인하였다.
    - 모델 학습이 진행되면서 train loss와 validation loss가 감소하는 경향을 그래프를 통해 확인했으며, 실제 요약문에 있는 핵심 단어들이 요약 문장 안에 포함되었다.
- Extractive 요약을 시도해 보고 Abstractive 요약 결과과 함께 비교해 보았다.
    - 두 요약 결과를 문법완성도 측면과 핵심단어 포함 측면으로 나누어 비교하고 분석 결과를 표로 정리하여 제시하였다.

> **[과제] 과제 제출**
>
> 노트북 파일(.ipynb)로 과제를 수행 후 출력이 저장된 상태의 깃헙 파일 링크를 입력해 주세요.
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).